# Conversational RAG with Short-term and Long-term Memory

## Objectives:

- Maintain conversations across multiple turns

- Recall information from previous interactions

- Retrieve domain-specific knowledge

- Scale to long-term personalized interactions

To achieve this, we designed a Conversational Retrieval-Augmented Generation (RAG) System equipped with both:

- Short-Term Memory (STM) for multi-turn dialogue

- Vector-Based Long-Term Memory (LTM) for storing persistent knowledge

In [ ]:
!pip install langchain>=1.0.7 langchain-community>=0.4.1 langchain-openai>=1.0.3 langchain_groq>=1.0.1 langchain_google_genai>=3.0.3 langchain-chroma>=1.0.0 langchain-text-splitters>=1.0.0 html2text>=2025.4.15

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 1.17.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.38.0 which is incompatible.
google-adk 1.17.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.38.0 which is incompatible.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.38.0 which is incompatible.

In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

Mounted at /content/drive


In [ ]:
# Connect to LLM
import os
import langchain
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-3.5-turbo")
response = llm.invoke("What is the capital of France?")

print(f"answer is: {response.content}")

answer is: The capital of France is Paris.


## Build the vectordB from web scrapings

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') # Replace with your Groq

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# Load webpage
urls = ["https://lilianweng.github.io/posts/2023-06-23-agent/"]
loader = WebBaseLoader(urls)
docs = loader.load()

# HTML → text
html2text = Html2TextTransformer()
docs = html2text.transform_documents(docs)

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = splitter.split_documents(docs)

# Vectorstore
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

In [ ]:
results = vectorstore.similarity_search("What is a LLM agent?", k=3)
print("\n🔍 Top 3 matching chunks:")
# results
for i, doc in enumerate(results):
    print(f"\n[{i+1}] Content: {doc.page_content[:100]}...")
    print(f"\n[{i+1}] Metadata: {doc.metadata}...")
    # print(f"\n[{i+1}] Simillarity Score: {doc}...")
    print(f"=================================================================")



🔍 Top 3 matching chunks:

[1] Content: Proof-of-Concept Examples Challenges Citation References Building agents with LLM (large language mo...

[1] Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'language': 'en', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent 

## Chat Prompt for Convesations with LSTM

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful Q&A assistant. Use the context below and the conversation history
to answer the user's last question. If the answer is not in the context, say so.

Conversation history:
{chat_history}

Retrieved context:
{context}

User question:
{question}
""")

## RAG Pipeline

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

#Retriever + Formatting Pipeline
from operator import itemgetter



In [ ]:
def format_docs(docs):
    # convert list of Document objects → string
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {
        # send ONLY the question string → retriever → format docs
        "context": itemgetter("question") | retriever | RunnableLambda(format_docs),

        # send the question string into the prompt
        "question": itemgetter("question"),

        # send chat history array as string (RunnableWithMessageHistory injects this)
        "chat_history": itemgetter("chat_history"),
    }
    | prompt
    | llm
)

## Enable multi-turn Conversation with context awareness
- ChatMessageHistory and RunnableWithMessageHistory

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# in-memory session store
store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversation_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="question",        # the new user input
    history_messages_key="chat_history"   # pipeline expects {chat_history}
)

In [ ]:
session_id = "memory_test_01"

print("\n---------------- TURN 1 ----------------")
response1 = conversation_rag_chain.invoke(
    {"question": "What is an AI agent?"},
    config={"configurable": {"session_id": session_id}}
)
print("A1:", response1)

print("\n---------------- TURN 2 ----------------")
response2 = conversation_rag_chain.invoke(
    {"question": "How do LLM-based agents work?"},
    config={"configurable": {"session_id": session_id}}
)
print("A2:", response2)

print("\n---------------- TURN 3 ----------------")
response3 = conversation_rag_chain.invoke(
    {"question": "Did you remember my first question?"},
    config={"configurable": {"session_id": session_id}}
)
print("A3:", response3)

print("\n---------------- TURN 4 ----------------")
response4 = conversation_rag_chain.invoke(
    {"question": "Summarize everything we have discussed so far."},
    config={"configurable": {"session_id": session_id}}
)
print("A4:", response4)

print("\n---------------- TURN 5 ----------------")
response5 = conversation_rag_chain.invoke(
    {"question": "Based on our discussion, what should I learn next?"},
    config={"configurable": {"session_id": session_id}}
)
print("A5:", response5)

print("\n---------------- TURN 6 ----------------")
response6 = conversation_rag_chain.invoke(
    {"question": "What was my second question again?"},
    config={"configurable": {"session_id": session_id}}
)
print("A6:", response6)


---------------- TURN 1 ----------------


/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


A1: content='The retrieved context does not provide a specific definition of an AI agent. Therefore, I cannot answer that question based on the provided information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 454, 'total_tokens': 481, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_560af6e559', 'finish_reason': 'stop', 'logprobs': None} id='run--8cefb909-433f-47bd-ae6d-6fe472fae57c-0'

---------------- TURN 2 ----------------
A2: content="LLM-based agents work by utilizing a large language model (LLM) as their core controller. These agents are designed to perform tasks by leveraging the LLM's capabilities, which include generating well-written text and solving problems. In practice, this involves integrating the LLM with 

In [ ]:

def chat_with_rag(session_id="chat_user"):
    print("RAG Chatbot Ready! (type 'exit' or 'quit' to stop)")
    print("-----------------------------------------------------")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit", "bye"]:
            print("Bot: Goodbye!")
            break

        # invoke the chain with memory
        response = conversation_rag_chain.invoke(
            {"question": user_input},
            config={"configurable": {"session_id": session_id}},
        )

        print(f"Bot: {response}\n")


# start chatbot
chat_with_rag()

RAG Chatbot Ready! (type 'exit' or 'quit' to stop)
-----------------------------------------------------
You: exit
Bot: Goodbye!


In [ ]:
session_id="chat_user" #"run--8eb56645-b013-4d1e-98b2-66e1f123dd86-0"

chat_with_rag(session_id)


RAG Chatbot Ready! (type 'exit' or 'quit' to stop)
-----------------------------------------------------
You: what's my name
Bot: content='Your name is Bibhu.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 1468, 'total_tokens': 1474, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_560af6e559', 'finish_reason': 'stop', 'logprobs': None} id='run--d780ff48-ed4d-43a1-a05d-f4ea60d42c10-0'

You: exit
Bot: Goodbye!


## RAG with long-term memory

In [24]:
# Create a Chroma dB as Long Term Memory Store
#Create vector Store

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

ltm_embeddings = OpenAIEmbeddings()

# Create a separate Chroma DB for long-term memory
ltm_vectorstore = Chroma(
    collection_name="long_term_memory",
    embedding_function=ltm_embeddings
)

# Long-term memory retriever
ltm_retriever = ltm_vectorstore.as_retriever(search_kwargs={"k": 3})


/tmp/ipython-input-1677433964.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  ltm_vectorstore = Chroma(


In [ ]:
#Long Term Memory Updation

def save_to_ltm(question, answer):
    """Store important conversational info as long-term memory."""
    text = f"User asked: {question}\nAssistant answered: {answer}"
    ltm_vectorstore.add_texts([text])

In [25]:
def join_text(data):
    return "\n\n".join([d.page_content for d in data])

rag_chain = (
    {
        "rag_context": itemgetter("question") | retriever | RunnableLambda(format_docs),
        "ltm_context": itemgetter("question") | ltm_retriever | RunnableLambda(format_docs),
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | ChatPromptTemplate.from_template("""
You are a helpful AI assistant. Use ALL sources below:

1. Short-term chat history:
{chat_history}

2. Long-term memory (LTM):
{ltm_context}

3. Retrieved knowledge (RAG):
{rag_context}

Answer the final user question:
{question}
""")
    | llm
)


In [26]:
conversation_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

In [27]:
def chatbot_fn(message, history):
    session_id = "memory_test_02"  #New with LMT

    answer = conversation_rag_chain.invoke(
        {"question": message},
        config={"configurable": {"session_id": session_id}},
    )

    # Save important info into long-term memory
    save_to_ltm(message, answer)

    return answer

In [28]:
# Test1
session_id = "memory_test_02"

conversation_rag_chain.invoke(
    {"question": "My favorite topic is Reinforcement Learning."},
    config={"configurable": {"session_id": session_id}},
)

# LTM now contains this fact
conversation_rag_chain.invoke(
    {"question": "What do I like to learn about?"},
    config={"configurable": {"session_id": session_id}},
)

AIMessage(content="You like to learn about Reinforcement Learning (RL), which is a fascinating area of machine learning that focuses on how agents can take actions in an environment to maximize cumulative rewards. If you're interested in exploring more about recent developments in this field or specific applications, feel free to ask!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 952, 'total_tokens': 1008, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None}, id='run--6ade6280-2f21-4e2b-8997-05011cbcde16-0')

In [29]:
session_id = "long_term_test_01"

print("=== TURN 1: Store a personal fact in LTM ===")
response1 = conversation_rag_chain.invoke(
    {"question": "My birthday is on 14th October."},
    config={"configurable": {"session_id": session_id}}
)
print("A1:", response1)

# Save to LTM
save_to_ltm("My birthday is on 14th October.", response1)


print("\n=== TURN 2–5: Distract the model with unrelated queries ===")
unrelated_questions = [
    "Explain the difference between supervised and unsupervised learning.",
    "What is the capital of France?",
    "Tell me something about reinforcement learning.",
    "What is backpropagation?"
]

for i, q in enumerate(unrelated_questions, start=2):
    print(f"\n--- TURN {i}: {q} ---")
    r = conversation_rag_chain.invoke(
        {"question": q},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"A{i}:", r)
    # these do NOT go to LTM — only distract



print("\n=== TURN 6: Critical: Ask LTM-dependent question ===")
response6 = conversation_rag_chain.invoke(
    {"question": "When is my birthday?"},
    config={"configurable": {"session_id": session_id}}
)
print("A6:", response6)


print("\n=== TURN 7: Even stronger LTM recall test ===")
response7 = conversation_rag_chain.invoke(
    {"question": "Earlier you learned a fact about my personal life. What was it?"},
    config={"configurable": {"session_id": session_id}}
)
print("A7:", response7)


print("\n=== TURN 8: Ask for reasoning using long-term memory ===")
response8 = conversation_rag_chain.invoke(
    {"question": "Since my birthday is on that date, what zodiac sign am I?"},
    config={"configurable": {"session_id": session_id}}
)
print("A8:", response8)


print("\n=== TURN 9: Ask to explain how it remembered ===")
response9 = conversation_rag_chain.invoke(
    {"question": "How did you remember my birthday even after many unrelated questions?"},
    config={"configurable": {"session_id": session_id}}
)
print("A9:", response9)


=== TURN 1: Store a personal fact in LTM ===
A1: content='Happy early birthday! If you have any plans or special ways you like to celebrate, feel free to share!' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 337, 'total_tokens': 359, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--a59b4dfe-4ecd-4896-861b-6c44c325c18a-0'

=== TURN 2–5: Distract the model with unrelated queries ===

--- TURN 2: Explain the difference between supervised and unsupervised learning. ---
A2: content='Supervised and unsupervised learning are two fundamental types of machine learning.\n\n1. **Supervised Learning**: In supervised learning, the model is trained on a labeled dataset, which 